# Day 4-03｜籃球軌跡追蹤與出手點估計
> Python 籃球運動資料分析課程  
> 用簡化顏色追蹤建立球軌跡，再討論真實影片中的限制。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 選擇學生影片或範例影片。
- 取得籃球中心點的 frame-by-frame 座標。
- 用軌跡速度估計出手 frame。

## 完成產出
- 球軌跡 CSV 與軌跡圖。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 選擇影片來源。
2. 執行顏色追蹤。
3. 儲存軌跡並判讀出手點。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
from src.video_utils import pick_first_converted_video
from src.shooting_utils import track_orange_ball, estimate_release_frame
from src.plot_utils import plot_ball_path

# 若學生沒有上傳影片，會自動使用 assets/samples/sample_ball_motion.mp4。
video_path = pick_first_converted_video(COURSE_ROOT)
print("video_path:", video_path)

In [ ]:
ball_df = track_orange_ball(video_path, max_frames=180)
print("tracked points:", len(ball_df))
ball_df.head()

In [ ]:
out_csv = COURSE_ROOT / "assets" / "results" / "d4_03_ball_track.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
ball_df.to_csv(out_csv, index=False)
print("saved:", out_csv)

plot_ball_path(
    ball_df, output_path=COURSE_ROOT / "assets" / "results" / "d4_03_ball_path.png"
)
release_frame = estimate_release_frame(ball_df)
print("estimated release frame:", release_frame)

## 討論

簡單顏色追蹤在下列情況會失敗：

- 球被手擋住。
- 球跟背景顏色太接近。
- 室外光線變化大。
- 影片壓縮太嚴重。

進階做法可以改成：YOLO ball detector、手動標幾個 frame、或用 tracking + interpolation。